In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python Batch

# ¿Cuándo se dispara el robot? Cuando hacemos un push a la rama main
on:
  push:
    branches: [ main ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest # Usará una máquina Linux en la nube para el ensamble

    steps:
    # Paso 1: Descargar el código del repositorio de GitHub
    - name: Descargar código
      uses: actions/checkout@v4

    # Paso 2: Configurar el entorno de Python en la máquina virtual
    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    # Paso 3: Instalar dependencias del proyecto
    - name: Instalar dependencias
      run: |
        python -m pip install --upgrade pip
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    # Paso 4: Validar la sintaxis del código (Linting)
    - name: Validar sintaxis con Flake8
      run: |
        pip install flake8
        # Detiene la ejecución si hay errores de sintaxis de Python
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

In [ ]:
if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "prod-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "prod-fleet-alerts")
    
    # LEER LA VARIABLE EXPLÍCITA DE SIMULACIÓN DESDE DOCKER COMPOSE
    SIMULACION = os.getenv("SIMULACION", "False") == "True"
    
    if not AZURE_CONNECTION_STRING or SIMULACION:
        print("⚠️  Advertencia: Modo de Simulación de Infraestructura Local Activo.")
        SIMULACION = True
        
    s3, azure_container_client = None, None
    if not SIMULACION:
        aws_enterprise_config = Config(
            connect_timeout=5,
            read_timeout=10,
            retries={'max_attempts': 3, 'mode': 'standard'}
        )
        s3 = boto3.client("s3", config=aws_enterprise_config)
        azure_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
        azure_container_client = azure_service_client.get_container_client(AZURE_CONTAINER)
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")
viejo

In [ ]:
if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "raw_telemetry.csv")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")
    
    # 1. LEER LA CONFIGURACIÓN DE DOCKER COMPOSE
    # Si pusiste SIMULACION=False en el YAML, aquí se evaluará como False.
    SIMULACION = os.getenv("SIMULACION", "True").upper() == "TRUE"
    
    s3, azure_container_client = None, None
    
    if not SIMULACION:
        # =========================================================
        # MODO PROCESAMIENTO LOCAL MASIVO (1,000 FILAS)
        # =========================================================
        print("⚡ [Modo Enterprise Local] Conectando a servicios simulados internos...")
        
        # Simulador local de AWS S3 en memoria (Moto)
        # Nota: Aquí asumimos que tu script inicializa el mock_aws antes o usa la librería local.
        # Para que no truene buscando internet, usamos un cliente local apuntando al ambiente interno
        s3 = boto3.client("s3", region_name="us-east-1")
        
        # Simulador local para Azure (Mock) para evitar usar la cadena de conexión real de internet
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                print(f"☁️  [Local Container] Interceptado exitosamente reporte '{name}' en simulación local.")
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        # =========================================================
        # MODO PRUEBA RÁPIDA (3 FILAS EN MEMORIA)
        # =========================================================
        print("⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.")
    
    print("📥 Connecting to S3 Intake Stream...")
    # Ahora sí, si SIMULACION es False, pasará False aquí y activará el "else" de tu streamer
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")

nuevo

In [ ]:
import io
import os
import random
from datetime import datetime, timedelta
from pathlib import Path
import boto3
from moto import mock_aws

# ==========================================
# CONFIGURACIÓN GENERAL Y PATHLIB
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
FLEET_DATA_DIR = BASE_DIR / "fleet_data"

print("🔧 Step 1: Checking and preparing local workbench folders...")

# Usamos Pathlib para asegurar que la carpeta exista de forma segura
FLEET_DATA_DIR.mkdir(parents=True, exist_ok=True)
local_raw_path = FLEET_DATA_DIR / "raw_fleet_telemetry.csv"

# A. Fabricar el dataset de 1,000 filas automáticamente
print("📝 Manufacturing fresh diagnostic data rows...")
with open(local_raw_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    start_time = datetime.now()
    fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE"]
    
    for i in range(1000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(100, 1006)}"
        rpm = random.randint(1500, 3500)
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"✅ Telemetry data file successfully secured at: {local_raw_path}\n")


print("⚡ Step 2: Igniting the Multi-Cloud Local Simulator...")

# Iniciar la simulación fresca de AWS Moto en memoria
aws_mock = mock_aws()
aws_mock.start()

# Configurar el entorno virtual de AWS S3
s3_client = boto3.client("s3", region_name="us-east-1")
s3_bucket_name = "fleet-raw-data"
s3_client.create_bucket(Bucket=s3_bucket_name)

# Subir nuestro CSV al S3 virtual (Convertimos el Path de Pathlib a string)
s3_client.upload_file(str(local_raw_path), s3_bucket_name, "raw_telemetry.csv")
print("☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'")


# ==========================================
# SIMULADORES DE AZURE (MOCKS)
# ==========================================
class MockAzureContainerClient:
    def __init__(self):
        self.stored_blobs = {}
    def upload_blob(self, name, data, overwrite=True):
        self.stored_blobs[name] = data
        print(f"☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob '{name}'!")

class MockBlobServiceClient:
    def __init__(self):
        pass
    def get_container_client(self, container_name):
        print(f"☁️ Microsoft Azure Simulation: Connected to container '{container_name}'")
        return MockAzureContainerClient()
        
    @classmethod
    def from_connection_string(cls, conn_str):
        return cls()

# Inicializar el cliente simulado de Azure
azure_container_client = MockBlobServiceClient().get_container_client("fleet-clean-alerts")

print("⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!\n")


# ==========================================
# VÁLVULAS DE PROCESAMIENTO (ETL LOGIC)
# ==========================================

# 1. Entrada: NUEVA versión híbrida con simulación rápida integrada y streaming
def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    """Streams records directly out of AWS S3 or fallback to local simulated data."""
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        # Generamos el set corto de prueba en memoria sin requerir AWS S3
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        # Modo estándar: Se conecta y descarga la telemetría del AWS S3 simulado
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
        
    # Procesamiento línea por línea usando un buffer de memoria seguro (io.StringIO)
    stream_buffer = io.StringIO(s3_data)
    
    # Brincar la línea de encabezados (headers)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

# 2. Transformación: Filtro OBD-II de códigos de falla críticos
def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

# 3. Salida: Compilación final y escritura acumulada hacia Azure
def azure_bulk_writer(container_client, blob_name, processed_records):
    """Compiles clean data rows and uploads them as a unified blob file to Azure."""
    if not processed_records:
        print("ℹ️ No critical incidents to write to Azure.")
        return
    
    # Construcción de encabezado y acumulación por fila usando +=
    csv_buffer = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)


# ==========================================
# EJECUCIÓN DEL PIPELINE (BATCH PRINCIPAL)
# ==========================================

# --- CONFIGURACIÓN DE PRUEBA DE LA ENTRADA ---
# Puedes cambiar simulation_mode a True si quieres probar solo tus 3 filas locales de memoria.
# Si está en False, leerá de forma automatizada las 1,000 líneas cargadas en el S3 simulado.
USAR_SIMULACION_RAPIDA = False

print("📥 Connecting to AWS S3 Intake Stream...")
data_stream = s3_telemetry_streamer(
    s3_client=s3_client, 
    bucket=s3_bucket_name, 
    key="raw_telemetry.csv", 
    simulation_mode=USAR_SIMULACION_RAPIDA
)

incidents = []
for raw_row in data_stream:
    clean_row = telemetry_processor(raw_row)
    if clean_row:
        incidents.append(clean_row)
        
print(f"📤 Uploading {len(incidents)} processed records to Azure Blob Storage...")
# Ejecución final del reporte compilado hacia nuestro cliente Azure simulado
azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents)

print("🏁 Batch complete! Multi-cloud handoff and local auditing successful.")

# Apagar limpiamente el simulador en memoria al finalizar
aws_mock.stop()

In [1]:
%%javascript
// This script automatically locks the current active notebook cell
var cell = Jupyter.notebook.get_selected_cell();
cell.metadata.editable = false; // Prevents editing the code
cell.metadata.deletable = false; // Prevents accidental deletion ('DD')
alert("🔒 Cell successfully locked! It cannot be modified or deleted.");

<IPython.core.display.Javascript object>

In [ ]:
###########################################################################################

In [ ]:
4. Refrigerants Used in Home Refrigerators
Refrigerants are the "working fluid" that circulates through the refrigeration system, absorbing and releasing heat.Historically, various refrigerants have been used, but due to environmental concerns (ozone depletion and global warming potential), regulations have led to changes:


CFCs (Chlorofluorocarbons) - e.g., R-12 (Freon-12): These were widely used but were phased out globally due to their high ozone-depleting potential (ODP). You won't find new refrigerators using R-12.

HCFCs (Hydrochlorofluorocarbons) - e.g., R-22: These were transitional refrigerants, less harmful to the ozone layer than CFCs but still with some ODP. Their production is also largely phased out for new equipment.


HFCs (Hydrofluorocarbons) - e.g., R-134a: This is currently one of the most common refrigerants used in home refrigerators. It has zero ODP but does have a global warming potential (GWP), contributing to climate change. Many newer refrigerators use R-134a.



Hydrocarbons (HCs) - e.g., R-600a (Isobutane), R-290 (Propane): These are becoming increasingly popular, especially in Europe and newer models globally. They have very low GWP and zero ODP, making them environmentally friendly. They are flammable, so systems using them are designed with safety in mind (e.g., smaller charges).


HFOs (Hydrofluoroolefins) - e.g., R-1234yf: These are newer generation refrigerants with extremely low GWP, designed as replacements for HFCs in various applications. They are becoming more common.



5. Basic Certification Needed to Work on Home Refrigerators Related to Refrigerants
In the United States, to work on any appliance that could release ozone-depleting or high-GWP substitute refrigerants into the atmosphere, you generally need to be certified under Section 608 of the Clean Air Act by the U.S. Environmental Protection Agency (EPA).
For home refrigerators (which are often categorized as "small appliances" if they are fully charged with 5 pounds or less of refrigerant), the basic certification required is Type I Certification.

Type I (Small Appliances): This certification allows you to service small appliances, which typically include household refrigerators and freezers, window air conditioners, and water coolers.


Universal Certification: This covers all types of refrigeration equipment (Type I, Type II - high-pressure, and Type III - low-pressure), so if you have a Universal certification, you are also covered for home refrigerators.


What does this certification entail?
It requires passing an EPA-approved test that covers proper refrigerant handling techniques, including recovery, recycling, and reclamation to prevent harmful releases into the atmosphere.
The test covers core knowledge about refrigerants, ozone depletion, global warming, and safe handling practices, along with specific knowledge for the type of equipment (e.g., small appliances).


Why is it needed?
It's a federal requirement to protect the environment. Refrigerants, especially older ones like CFCs and HCFCs, can deplete the ozone layer. Even newer HFCs, while not ozone-depleting, are potent greenhouse gases.
This certification ensures technicians understand how to work with refrigerants responsibly, preventing leaks and ensuring proper disposal and recycling.
You cannot legally purchase or handle refrigerants for refrigeration systems without this certification. Many trade schools and HVAC/R training programs offer courses that prepare you for the EPA Section 608 exam.
Sources 


Playing Detective: Finding Those Sneaky Leaks
Part of responsible refrigerant handling is finding and fixing leaks. Here are some common ways:
Bubble Test (Soap Bubbles): This is the classic! You apply a soapy solution to fittings and joints. If you see bubbles forming, you've found a leak! It's simple, but effective for larger leaks.
Electronic Leak Detectors: These are handheld gadgets that "sniff" the air for traces of refrigerant. They'll usually beep faster or light up more intensely as you get closer to a leak. They're great for smaller, harder-to-find leaks.
Fluorescent Dye: A special dye is injected into the system. If there's a leak, the dye will escape with the refrigerant, and you can spot it using a UV (ultraviolet) light.
Ultrasonic Leak Detectors: These devices listen for the high-frequency sound of gas escaping a small opening. They're good for loud environments where other methods might be hard to hear.

Your Basic Toolkit (Just a Glimpse!)
To give you an idea of what a technician might use:
Manifold Gauges: These are the essential tools for reading the pressure in a refrigeration system, helping you diagnose how it's running.
Recovery Machine: The device that pumps refrigerant out of a system into a recovery tank.
Vacuum Pump: Used to remove air and moisture from a system after repairs, before adding new refrigerant.
Multimeter: For checking electrical voltage, current, and continuity.
Wrenches, Screwdrivers, Pliers: The usual suspects for disassembly and reassembly.

In [ ]:
4. Refrigerants Used in Home Refrigerators
Refrigerants are the "working fluid" that circulates through the refrigeration system, absorbing and releasing heat.Historically, various refrigerants have been used, but due to environmental concerns (ozone depletion and global warming potential), regulations have led to changes:


CFCs (Chlorofluorocarbons) - e.g., R-12 (Freon-12): These were widely used but were phased out globally due to their high ozone-depleting potential (ODP). You won't find new refrigerators using R-12.

HCFCs (Hydrochlorofluorocarbons) - e.g., R-22: These were transitional refrigerants, less harmful to the ozone layer than CFCs but still with some ODP. Their production is also largely phased out for new equipment.


HFCs (Hydrofluorocarbons) - e.g., R-134a: This is currently one of the most common refrigerants used in home refrigerators. It has zero ODP but does have a global warming potential (GWP), contributing to climate change. Many newer refrigerators use R-134a.



Hydrocarbons (HCs) - e.g., R-600a (Isobutane), R-290 (Propane): These are becoming increasingly popular, especially in Europe and newer models globally. They have very low GWP and zero ODP, making them environmentally friendly. They are flammable, so systems using them are designed with safety in mind (e.g., smaller charges).


HFOs (Hydrofluoroolefins) - e.g., R-1234yf: These are newer generation refrigerants with extremely low GWP, designed as replacements for HFCs in various applications. They are becoming more common.



5. Basic Certification Needed to Work on Home Refrigerators Related to Refrigerants
In the United States, to work on any appliance that could release ozone-depleting or high-GWP substitute refrigerants into the atmosphere, you generally need to be certified under Section 608 of the Clean Air Act by the U.S. Environmental Protection Agency (EPA).
For home refrigerators (which are often categorized as "small appliances" if they are fully charged with 5 pounds or less of refrigerant), the basic certification required is Type I Certification.

Type I (Small Appliances): This certification allows you to service small appliances, which typically include household refrigerators and freezers, window air conditioners, and water coolers.


Universal Certification: This covers all types of refrigeration equipment (Type I, Type II - high-pressure, and Type III - low-pressure), so if you have a Universal certification, you are also covered for home refrigerators.


What does this certification entail?
It requires passing an EPA-approved test that covers proper refrigerant handling techniques, including recovery, recycling, and reclamation to prevent harmful releases into the atmosphere.
The test covers core knowledge about refrigerants, ozone depletion, global warming, and safe handling practices, along with specific knowledge for the type of equipment (e.g., small appliances).


Why is it needed?
It's a federal requirement to protect the environment. Refrigerants, especially older ones like CFCs and HCFCs, can deplete the ozone layer. Even newer HFCs, while not ozone-depleting, are potent greenhouse gases.
This certification ensures technicians understand how to work with refrigerants responsibly, preventing leaks and ensuring proper disposal and recycling.
You cannot legally purchase or handle refrigerants for refrigeration systems without this certification. Many trade schools and HVAC/R training programs offer courses that prepare you for the EPA Section 608 exam.
Sources 


Playing Detective: Finding Those Sneaky Leaks
Part of responsible refrigerant handling is finding and fixing leaks. Here are some common ways:
Bubble Test (Soap Bubbles): This is the classic! You apply a soapy solution to fittings and joints. If you see bubbles forming, you've found a leak! It's simple, but effective for larger leaks.
Electronic Leak Detectors: These are handheld gadgets that "sniff" the air for traces of refrigerant. They'll usually beep faster or light up more intensely as you get closer to a leak. They're great for smaller, harder-to-find leaks.
Fluorescent Dye: A special dye is injected into the system. If there's a leak, the dye will escape with the refrigerant, and you can spot it using a UV (ultraviolet) light.
Ultrasonic Leak Detectors: These devices listen for the high-frequency sound of gas escaping a small opening. They're good for loud environments where other methods might be hard to hear.

Your Basic Toolkit (Just a Glimpse!)
To give you an idea of what a technician might use:
Manifold Gauges: These are the essential tools for reading the pressure in a refrigeration system, helping you diagnose how it's running.
Recovery Machine: The device that pumps refrigerant out of a system into a recovery tank.
Vacuum Pump: Used to remove air and moisture from a system after repairs, before adding new refrigerant.
Multimeter: For checking electrical voltage, current, and continuity.
Wrenches, Screwdrivers, Pliers: The usual suspects for disassembly and reassembly.

Bring google keep qustions

In [ ]:
#################

In [ ]:
Recursion:

In [1]:
def countup(n, current=1):
 
    if current > n:   
        print("Finished!")
        return current
        
        
    else:
        print(current)
        countup(n, current + 1)
        

# Evaluation
countup(5)

1
2
3
4
5
Finished!


In [17]:
def countdown(n):
    if n <= 0:  
        print("Blastoff!")
        
    else:  
        print(n)
        countdown(n - 1)
        

countdown(2) 

2
1
Blastoff!


In [3]:
def countup_iterative(n):
 
    if n <= 0:
        print("Finished!")
        return

    for i in range(1, n + 1):# start, end, step diff from countdown
        print(i)
    print("Finished!")

# Evaluation
countup_iterative(5)
# countup_iterative(0)
# countup_iterative(-3)

1
2
3
4
5
Finished!


In [ ]:
def countdown_iterative(n):
    if n <= 0:
        print("Blastoff!")
        return

    for i in range(n, 0, -1):
        print(i)
    print("Blastoff!")

# Evaluation
countdown_iterative(5)
# countdown_iterative(0)
# countdown_iterative(-3)

In [13]:
def countdown(n):
    print(f"countdown({n}) called")   #Print when the function is called - 1 countdown(4) called
    if n <= 0:
        print("countdown base case reached") #print when base case is reached. - 3countdown base case reached

        print("Blastoff!") # 4 Blastoff!

        print(f"countdown({n}) returning on 0") #print when function returns. - 5 not to 8  countdown(0) returning

        print("I think I am understanding") #- 6 I think I am understanding

        print ('Yes I think so')  #- 7 Yes I think so
   
    else:
        print(n)
        print(f"countdown({n}) printing {n}") # - 2  print the number.  


         
        countdown(n - 1)
        print(f"countdown({n}) returning") # - 8 print when function returns. countdown(1,2,3,4) returning
    return ('I think I am really really understanding') #- 9 'I think I am really really understanding'

 
countdown(4)

countdown(4) called
4
countdown(4) printing 4
countdown(3) called
3
countdown(3) printing 3
countdown(2) called
2
countdown(2) printing 2
countdown(1) called
1
countdown(1) printing 1
countdown(0) called
countdown base case reached
Blastoff!
countdown(0) returning on 0
I think I am understanding
Yes I think so
countdown(1) returning
countdown(2) returning
countdown(3) returning
countdown(4) returning


'I think I am really really understanding'

In [ ]:
@@@@@@@@bueno...entre a myjuryportal y est